In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
import requests
from shapely.geometry import LineString, box

# ---- CRS ----
CRS_WGS84 = "EPSG:4326"
CRS_BNG   = "EPSG:27700"

# ---- Inputs ----
STATIONS_CSV = "glasgow_stations.csv"
TRIPS_CSV    = "glatrips.csv"
INFRA_SHP    = "CyclingRoutes/Cycling_Routes_Open.shp"

# ---- Shapefile fields:
TYPE_FIELD   = "PUB_CLASS"    
SCHEME_FIELD = "EXTRA_INFO"
DATE_FIELD   = None           

# ---- Parameters ----
SNAP_TOL_M  = 15
INFRA_TYPES = ["segregated", "lane", "shared", "mixed"]
ANY_TYPES   = ["segregated", "lane", "shared"]
ROUTER      = "osmnx_shortest"
PLACE_MARGIN_DEG = 0.02
OVERPASS_ENDPOINT = "https://overpass-api.de/api/interpreter"   # appendix only
CYCLESTREETS_API_KEY = ""

# ---- Study window ----
# Epochs are derived automatically from the unique 'opened' dates in the
# infrastructure shapefile (see §8). No hand-coded breakpoints needed.
STUDY_START = pd.Timestamp("2017-09-15")   # first date of your trip data window
STUDY_END   = pd.Timestamp("2024-04-01")   # last date of your trip data window

# ---- EXTRA_INFO spelling variants -> canonical (one date then covers both spellings) ----
EXTRA_INFO_NORMALISE = {
    "South West City Way":    "South-West City Way",
    "Forth and Clyde Canal":  "Forth & Clyde Canal",
    "Kelvingrove Park Route": "Kelvingrove Park",
    "Cycle marking on road":  "Cycle Marking on Road",
    "Park route":             "Park Route",
}

# ---- EXTRA_INFO value -> opening date. FILL the datable schemes as you find dates. ----
# Anything not in here (blank EXTRA_INFO, park/canal routes, descriptors) defaults to STUDY_START
# = "present in every regime". That is CORRECT for genuinely old network, WRONG for recent
# light-segregation materials -> see the SfP block. The §5b diagnostic flags the risky ones.
SCHEMES_OPENED = {
    "Connecting Woodside":            pd.Timestamp("2021-06-06"),
    # ---- Spaces for People light segregation: DO NOT let these default to always-on ----
    "Armadillo":                      pd.Timestamp("2020-09-13"), #glasgow.gov.uk/article/5464/Ruchill-Bilsland-Drive
}

# ---- Phased schemes: date segments by a SUB-FIELD, not the scheme name ----
SCW_STREET_OPENED = {
    # street (ROUTENAMEO among SCW rows) -> phase opening date
    "Victoria Rd":      pd.Timestamp("2018-11-14"),  # OBJECTID 812,1056,1096,1537  (south)
    "Allison St":     pd.Timestamp("2018-11-14"),  # 1885,1886
    "Calder St":      pd.Timestamp("2018-11-14"),  # 1887,1888
    "Pollokshaws Rd": pd.Timestamp("2019-03-29"),  # 586,1057,1058,1059,1097  (middle)
    "Gorbals St":     pd.Timestamp("2023-04-03"),  # 1095   final leg
    "Bridgegate":     pd.Timestamp("2024-06-17"),  # 1708   final leg
    "King Street":    pd.Timestamp("2024-06-17"),  # 543    final leg (northernmost)
}
PHASED_SCHEMES = {
    # scheme (normalised EXTRA_INFO) -> date its segments by this sub-field
    "South City Way": {"field": "ROUTENAMEO", "dates": SCW_STREET_OPENED},
}
# Surgical override: OBJECTID -> date (beats everything; use where one street spans two phases)
SEGMENT_OPENED = {
    1917 : pd.Timestamp("2020-07-17"), # Spaces for People London Road
    1918 : pd.Timestamp("2020-07-17"), # Spaces for People (glasgow.gov.uk/article/5412/London-Road-Phase-2-MacDuff-Street-to-London-Road-Shell-Garage)
    1192 : pd.Timestamp("2020-07-17"), # Spaces for People London Road
    1195 : pd.Timestamp("2020-07-17"), # Spaces for People (glasgow.gov.uk/article/5412/London-Road-Phase-2-MacDuff-Street-to-London-Road-Shell-Garage)
    1536 : pd.Timestamp("2023-07-06"), # East City Way
    1515: pd.Timestamp("2023-07-06"), # East City Way
    1533: pd.Timestamp("2023-07-06"), # East City Way
    1532: pd.Timestamp("2023-07-06"), # East City Way
    776: pd.Timestamp("2023-07-06"), # East City Way (walkwheelcycletrust.org.uk/our-blog/news/celebrations-as-major-glasgow-cycling-route-reaches-city-centre/)
    1916: pd.Timestamp("2023-07-06"), # East City Way
    1915: pd.Timestamp("2023-07-06"), # East City Way
    694: pd.Timestamp("2023-07-06"), # East City Way
    
    822: pd.Timestamp("2025-06-01"), # George V Bridge
    460: pd.Timestamp("2025-06-01"), # George V Bridge
    
    982: pd.Timestamp("2024-09-07"), # Govan Partick Bridge
    1919: pd.Timestamp("2024-09-07"), # Govan Partick Bridge
    
    1723: pd.Timestamp("2020-10-02"), # Hawthorn Street glasgow.gov.uk/article/5466/Possil-Park-Hawthorn-Street
    1719: pd.Timestamp("2020-10-02"), # Hawthorn Street
    1218: pd.Timestamp("2020-10-02"), # Hawthorn Street
    1201: pd.Timestamp("2020-10-02"), # Hawthorn Street
    1718: pd.Timestamp("2020-10-02"), # Hawthorn Street
    1722: pd.Timestamp("2020-10-02"), # Hawthorn Street
    
    1946: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave (glasgow.gov.uk/article/5410/Great-Western-Rd-Duntreath-Ave-to-Lincoln-Ave)
    1947: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1959: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1958: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1199: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1948: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1960: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1957: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1949: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1956: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1950: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1955: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1954: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1951: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1953: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1194: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    1952: pd.Timestamp("2020-07-21"), # Great Western Road: Duntreath Ave - Lincoln Ave
    
    578: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road (www.glasgow.gov.uk/article/5429/Cumbernauld-Road-Station-Road-to-Provanmill-Road)
    577: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road
    1734: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road
    635: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road
    1735: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road
    639: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road
    1733: pd.Timestamp("2020-08-20"), # Cumbernauld Rd: Station Rd to Provanmill Road
    
    579: pd.Timestamp("2020-08-20"), # Provanmill Rd: Cumernauld Rd to Royston Road (glasgow.gov.uk/article/5431/Provanmill-Road-Cumbernauld-Road-to-Royston-Road)
    638: pd.Timestamp("2020-08-20"), # Provanmill Rd: Cumernauld Rd to Royston Road
    
    1713: pd.Timestamp("2020-05-15"), #Broomielaw: Saltmarket to Clyde Arc (glasgow.gov.uk/article/5413/City-Centre-Broomielaw)
    1198: pd.Timestamp("2020-05-15"), #Broomielaw: Saltmarket to Clyde Arc
    1219: pd.Timestamp("2020-05-15"), #Broomielaw: Saltmarket to Clyde Arc
    1220: pd.Timestamp("2020-05-15"), #Broomielaw: Saltmarket to Clyde Arc
    1907: pd.Timestamp("2020-05-15"), #Broomielaw: Saltmarket to Clyde Arc
    1216: pd.Timestamp("2020-05-15"), #Broomielaw: Saltmarket to Clyde Arc
    
    1368: pd.Timestamp("2020-10-02"), #Corkerhill Road (glasgow.gov.uk/article/5471/Corkerhill-Corkerhill-Road)
    1367: pd.Timestamp("2020-10-02"), #Corkerhill Road
    
    1584: pd.Timestamp("2020-09-29"), #Kelvinway (glasgow.gov.uk/article/5514/Kelvingrove-Kelvin-Way)
    1767: pd.Timestamp("2020-09-29"), #Kelvinway
    
    1425: pd.Timestamp("2020-12-04"), #Hyndland (glasgow.gov.uk/article/5572/Hyndland-Clarence-Drive)
    1429: pd.Timestamp("2020-12-04"), #Hyndland
    1431: pd.Timestamp("2020-12-04"), #Hyndland
    1428: pd.Timestamp("2020-12-04"), #Hyndland
    1430: pd.Timestamp("2020-12-04"), #Hyndland
    1426: pd.Timestamp("2020-12-04"), #Hyndland
    1435: pd.Timestamp("2020-12-04"), #Hyndland
    1427: pd.Timestamp("2020-12-04"), #Hyndland
    1432: pd.Timestamp("2020-12-04"), #Hyndland
    1752: pd.Timestamp("2020-12-04"), #Hyndland
    1753: pd.Timestamp("2020-12-04"), #Hyndland
    1754: pd.Timestamp("2020-12-04"), #Hyndland
    
    1338: pd.Timestamp("2021-03-21"), #Brockburn Road (glasgow.gov.uk/article/5573/Pollok-Brockburn-Road)
    1319: pd.Timestamp("2021-03-21"), #Brockburn Road
    1347: pd.Timestamp("2021-03-21"), #Brockburn Road
    1342: pd.Timestamp("2021-03-21"), #Brockburn Road
    1348: pd.Timestamp("2021-03-21"), #Brockburn Road
    1349: pd.Timestamp("2021-03-21"), #Brockburn Road
    1341: pd.Timestamp("2021-03-21"), #Brockburn Road
    1339: pd.Timestamp("2021-03-21"), #Brockburn Road
    1317: pd.Timestamp("2021-03-21"), #Brockburn Road
    1343: pd.Timestamp("2021-03-21"), #Brockburn Road
    1340: pd.Timestamp("2021-03-21"), #Brockburn Road
    1345: pd.Timestamp("2021-03-21"), #Brockburn Road
    1337: pd.Timestamp("2021-03-21"), #Brockburn Road
    1346: pd.Timestamp("2021-03-21"), #Brockburn Road
    1321: pd.Timestamp("2021-03-21"), #Brockburn Road
    1318: pd.Timestamp("2021-03-21"), #Brockburn Road
    1322: pd.Timestamp("2021-03-21"), #Brockburn Road
    1323: pd.Timestamp("2021-03-21"), #Brockburn Road
    1320: pd.Timestamp("2021-03-21"), #Brockburn Road
    1324: pd.Timestamp("2021-03-21"), #Brockburn Road
    
    1844: pd.Timestamp("2020-09-01"), # Argyle Street
    
    85: pd.Timestamp("2021-02-22"), # Howard Street (glasgow.gov.uk/article/2870/Spaces-for-People-to-Improve-City-Centre-Cycle-Lane)
    
    636: pd.Timestamp("2021-05-01"), # Royston Rd (glasgow.gov.uk/article/3861/Improved-Active-Travel-Measures-for-Royston-Road)
    634: pd.Timestamp("2021-05-01"), # Royston Rd
    1839: pd.Timestamp("2021-05-01"), # Royston Rd
    1838: pd.Timestamp("2021-05-01"), #Royston Rd
    
    450: pd.Timestamp("2021-04-27"), # Wallacewell Rd (glasgow.gov.uk/article/2863/Spaces-for-People-Active-Travel-Improvements-for-Balornock) 
    448: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    1874: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    1979: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    1980: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    1981: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    1582: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    1583: pd.Timestamp("2021-04-27"), # Wallacewell Rd
    
    1386: pd.Timestamp("2020-05-15"), # Dumbreck St (glasgow.gov.uk/article/5411/Dumbreck-Road)
    
    1363: pd.Timestamp("2021-04-10"), # Cambridge Street (glasgow.gov.uk/article/3868/Improvements-to-Cambridge-St-Cycling-Lanes-Starting-Soon)
    
    882: pd.Timestamp("2018-08-01"), #BMX Scheme Linclon Ave & Archerhill Road (gobike.org/page/58?subscribe=success&ref=gobike.ghost.io)
    884: pd.Timestamp("2018-08-01"),
    885: pd.Timestamp("2018-08-01"),
    886: pd.Timestamp("2018-08-01"),
    
    1849: pd.Timestamp("2024-10-01"), #Byres Road & Church Street not open during data collection
    1848: pd.Timestamp("2024-10-01"),
    1847: pd.Timestamp("2024-10-01"),
    
    1867: pd.Timestamp("2025-08-01"), #Cowcaddens Road post data collection (glasgow.gov.uk/article/12222/New-Avenues-Plus-project-at-Cowcaddens-Road-and-Dobbie-s-Loan-to-begin-in-Glasgow)
    
    570: pd.Timestamp("2023-02-01") #St Andrews Drive Cycle way (openstreetmap.org/way/1098920498/history/2)
    
    
    
 
}

pd.set_option("display.width", 140)
print("config loaded")

config loaded


## 2. Stations → GeoDataFrame + study polygon

In [2]:
stations = pd.read_csv(STATIONS_CSV).drop_duplicates("station_id").reset_index(drop=True)
assert {"station_id","lat","lon"}.issubset(stations.columns), stations.columns.tolist()
stations = gpd.GeoDataFrame(stations,
    geometry=gpd.points_from_xy(stations["lon"], stations["lat"]), crs=CRS_WGS84)
stations_bng = stations.to_crs(CRS_BNG)

minx, miny, maxx, maxy = stations.total_bounds
STUDY_POLY_4326 = box(minx-PLACE_MARGIN_DEG, miny-PLACE_MARGIN_DEG,
                      maxx+PLACE_MARGIN_DEG, maxy+PLACE_MARGIN_DEG)
STUDY_BBOX_SWNE = (miny-PLACE_MARGIN_DEG, minx-PLACE_MARGIN_DEG,
                   maxy+PLACE_MARGIN_DEG, maxx+PLACE_MARGIN_DEG)
print(f"{len(stations)} stations")

120 stations


## 3. OD pairs to route


In [3]:
trips = pd.read_csv(TRIPS_CSV, parse_dates=["started_at"])
assert {"origin","destination","started_at"}.issubset(trips.columns), trips.columns.tolist()
known = set(stations["station_id"])
trips = trips[trips["origin"].isin(known) & trips["destination"].isin(known)]
trips = trips[trips["origin"] != trips["destination"]].reset_index(drop=True)

def canon(a, b):
    return (a, b) if a <= b else (b, a)

pairs = sorted({canon(o, d) for o, d in zip(trips["origin"], trips["destination"])})
print(f"{len(trips):,} trips -> {len(pairs):,} unordered pairs to route")

1,202,015 trips -> 5,906 unordered pairs to route


In [4]:
def build_graph():
    G = ox.graph_from_polygon(STUDY_POLY_4326, network_type="bike", simplify=True)
    return ox.project_graph(G, to_crs=CRS_BNG)

def route_osmnx(G, node_of):
    routes, failed = {}, []
    for a, b in pairs:
        r = ox.routing.shortest_path(G, node_of[a], node_of[b], weight="length")
        if r is None or len(r) < 2:
            failed.append((a, b)); continue
        edges = ox.routing.route_to_gdf(G, r, weight="length")
        routes[(a, b)] = (edges.geometry.union_all(), float(edges["length"].sum()))
    return routes, failed

if ROUTER == "osmnx_shortest":
    G = build_graph()
    nodes = ox.distance.nearest_nodes(G, X=stations_bng.geometry.x.values,
                                         Y=stations_bng.geometry.y.values)
    node_of = dict(zip(stations_bng["station_id"], nodes))
    routes, failed = route_osmnx(G, node_of)
else:
    raise ValueError(ROUTER)
print(f"routed {len(routes):,} pairs; {len(failed)} failed")

routed 5,904 pairs; 2 failed


## 5. Infrastructure from the GCC shapefile

In [5]:
_shp = gpd.read_file(INFRA_SHP)
print("CRS:", _shp.crs)
print("columns:", list(_shp.columns))
print("\n== TYPE_FIELD value_counts ==")
print(_shp[TYPE_FIELD].value_counts(dropna=False))
if SCHEME_FIELD:
    print("\n== SCHEME_FIELD value_counts (top 30) ==")
    print(_shp[SCHEME_FIELD].value_counts(dropna=False).head(30))
if DATE_FIELD:
    print("\n== DATE_FIELD sample ==")
    print(_shp[DATE_FIELD].dropna().head(10).tolist())

CRS: EPSG:27700
columns: ['OBJECTID', 'HYPERLINK', 'LOC_AUTHOR', 'CATEGORY_1', 'CATEGORY_2', 'CATEGORY_3', 'CATEGORY_4', 'CATEGORY_5', 'EXTRA_INFO', 'EXTRA_IN_1', 'ROUTENAMEO', 'TYPE_NEW', 'PUB_CLASS', 'SHAPELEN', 'geometry']

== TYPE_FIELD value_counts ==
PUB_CLASS
Shared Path              484
Segregated Cycle Lane    173
Cycle Lane               118
Mixed Traffic Street     116
Shared Footway            73
Name: count, dtype: int64

== SCHEME_FIELD value_counts (top 30) ==
EXTRA_INFO
NaN                               739
Armadillo                          48
Signage                            17
South City Way                     16
Scottish Outdoor Access Code       14
Sighthill TRA project              12
North East Active Travel Route     10
Kelvin Walkway                      9
Pollok Country Park                 8
Connecting Woodside                 8
Forth & Clyde Canal                 7
East City Way                       6
Cycle Marking on Road               6
Avenue B       

### 5b. Load, classify, and date every from the shp segment


In [6]:
# Map GCC PUB_CLASS -> analysis types. Confirmed exact-match against the .dbf value_counts.
GCC_TYPE_MAP = {
    "Segregated Cycle Lane": "segregated",
    "Cycle Lane":            "lane",
    "Shared Footway":        "shared",
    "Shared Path":           "shared",
    "Mixed Traffic Street":  "mixed",
}

def parse_opened(v):
    if pd.isna(v): return pd.NaT
    if isinstance(v, (int, float, np.integer, np.floating)) and 1900 <= v <= 2100:
        return pd.Timestamp(int(v), 1, 1)
    s = str(v).strip()
    if s.isdigit() and len(s) == 4:
        return pd.Timestamp(int(s), 1, 1)
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

def derive_opened(row):
    # 1. surgical per-segment override (OBJECTID)
    oid = row.get("OBJECTID")
    if pd.notna(oid):
        try:
            if int(oid) in SEGMENT_OPENED:
                return SEGMENT_OPENED[int(oid)]
        except (ValueError, TypeError):
            pass
    scheme = row.get(SCHEME_FIELD) if SCHEME_FIELD else None
    scheme = EXTRA_INFO_NORMALISE.get(scheme, scheme)     # fold spelling variants
    # 2. phased scheme: date by a sub-field (e.g. ROUTENAMEO) within the scheme
    if scheme in PHASED_SCHEMES:
        spec = PHASED_SCHEMES[scheme]
        d = spec["dates"].get(row.get(spec["field"]))
        if d is not None:
            return d
    # 3. explicit date field (none in this dataset)
    if DATE_FIELD and pd.notna(row.get(DATE_FIELD)):
        d = parse_opened(row[DATE_FIELD])
        if pd.notna(d): return d
    # 4. scheme-level single date
    if scheme is not None:
        d = SCHEMES_OPENED.get(scheme)
        if d is not None: return d
    # 5. default: predates the study window -> present in every regime
    return STUDY_START

def load_infra():
    g = gpd.read_file(INFRA_SHP).to_crs(CRS_BNG)
    g = g[g.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    g["infra_type"] = g[TYPE_FIELD].map(GCC_TYPE_MAP)
    unmapped = sorted(set(g.loc[g["infra_type"].isna(), TYPE_FIELD].dropna().unique()))
    if unmapped:
        print("WARNING unmapped TYPE_FIELD values (edit GCC_TYPE_MAP):", unmapped)
    g = g.dropna(subset=["infra_type"]).copy()
    g["opened"] = g.apply(derive_opened, axis=1)
    keep = ["infra_type", "opened", "geometry"]
    for c in [SCHEME_FIELD, "CATEGORY_4"]:          # kept for the diagnostic + optional hard/light split
        if c and c in g.columns and c not in keep:
            keep.append(c)
    return g[keep]

infra_gcc = load_infra()
print(infra_gcc["infra_type"].value_counts())

# ---- temporal diagnostic: dated-from-scheme vs defaulted-to-always-on ----
defaulted = infra_gcc["opened"].eq(STUDY_START)
print(f"\nsegments dated from a scheme: {int((~defaulted).sum())} | "
      f"defaulted to always-on: {int(defaulted.sum())}")

# flag likely Spaces-for-People light segregation that DEFAULTED (temporal-bias risk)
LIGHT = ["Armadillo", "Orcas", "Batons", "Segregation islands/batons", "Light segregation"]
is_light = pd.Series(False, index=infra_gcc.index)
if "CATEGORY_4" in infra_gcc.columns:
    is_light |= infra_gcc["CATEGORY_4"].isin(LIGHT)
if SCHEME_FIELD in infra_gcc.columns:
    is_light |= infra_gcc[SCHEME_FIELD].isin(LIGHT)
risky = int((is_light & defaulted).sum())
if risky:
    print(f"WARNING: {risky} light-segregation segments defaulted to always-on -> "
          f"likely Spaces-for-People era. Add the relevant keys to SCHEMES_OPENED with an SfP date.")


infra_type
shared        557
segregated    173
lane          118
mixed         116
Name: count, dtype: int64

segments dated from a scheme: 147 | defaulted to always-on: 817


## 6. Exposure core — run the self-test cell first

In [7]:
def build_infra_buffers(infra_gdf, types=INFRA_TYPES, any_types=None, snap_tol=SNAP_TOL_M):
    """One buffered polygon per type; 'any' = union over any_types only (mixed excluded by default)."""
    any_types = list(types) if any_types is None else any_types
    buffers = {}
    for t in list(types):
        sub = infra_gdf[infra_gdf["infra_type"] == t]
        buffers[t] = (sub.geometry.union_all().buffer(snap_tol, cap_style="flat", join_style="round")
                      if len(sub) else None)
    anysub = infra_gdf[infra_gdf["infra_type"].isin(any_types)]
    buffers["any"] = (anysub.geometry.union_all().buffer(snap_tol, cap_style="flat", join_style="round")
                      if len(anysub) else None)
    return buffers

def exposure_for_routes(routes, buffers, types=INFRA_TYPES):
    rows = []
    for (a, b), (geom, L) in routes.items():
        rec = {"origin_c": a, "dest_c": b, "route_len_m": L}
        for t in list(types) + ["any"]:
            buf = buffers.get(t)
            inter = geom.intersection(buf).length if (buf is not None and L > 0) else 0.0
            rec[f"len_{t}_m"] = inter
            rec[f"exp_{t}"]   = (inter / L) if L > 0 else np.nan
        rows.append(rec)
    return pd.DataFrame(rows)

print("exposure functions defined")

exposure functions defined


In [8]:
# ---- SELF-TEST: overlay maths on synthetic geometry ----
from shapely.geometry import LineString as _LS
_route = _LS([(0, 0), (1000, 0)])
_infra = gpd.GeoDataFrame({"infra_type": ["segregated"]},
                          geometry=[_LS([(200, 0), (500, 0)])], crs=CRS_BNG)
_buf = build_infra_buffers(_infra, ["segregated"], snap_tol=15)
_got = exposure_for_routes({("A","B"): (_route, _route.length)}, _buf, ["segregated"])["exp_segregated"].iloc[0]
print(f"exp_segregated = {_got:.4f} (expected ~0.30)")
assert abs(_got - 0.30) < 0.01, "overlay logic FAILED"
print("SELF-TEST PASSED")

exp_segregated = 0.3000 (expected ~0.30)
SELF-TEST PASSED


## 7. STATIC exposure

In [9]:
buffers_now = build_infra_buffers(infra_gcc, INFRA_TYPES, ANY_TYPES)
static_exp = exposure_for_routes(routes, buffers_now, INFRA_TYPES)

def attach_exposure(od_df, exp_df, o="origin", d="destination"):
    key = od_df[[o, d]].apply(lambda r: canon(r[o], r[d]), axis=1)
    od_df = od_df.assign(origin_c=[k[0] for k in key], dest_c=[k[1] for k in key])
    return od_df.merge(exp_df, on=["origin_c","dest_c"], how="left").drop(columns=["origin_c","dest_c"])

od = trips.groupby(["origin","destination"]).size().reset_index(name="trips")
attach_exposure(od, static_exp).to_csv("glasgow_exposure_static.csv", index=False)
print(static_exp[["origin_c","dest_c","exp_any","exp_segregated"]].head())

                             origin_c                                             dest_c   exp_any  exp_segregated
0  Alexandra Parade (West) - ELECTRIC  Alexandra Park (south entrance) Alexandra Para...  0.000000        0.000000
1  Alexandra Parade (West) - ELECTRIC                          Anderston Railway Station  0.113684        0.043786
2  Alexandra Parade (West) - ELECTRIC                                 Anniesland Library  0.026504        0.009067
3  Alexandra Parade (West) - ELECTRIC           Argyle Street Railway Station - ELECTRIC  0.042397        0.012128
4  Alexandra Parade (West) - ELECTRIC                             Barrowlands - ELECTRIC  0.000000        0.000000


## 8. FLUID exposure 

Each unique `opened` date in the infrastructure defines an epoch


In [10]:
def build_epochs(infra_gdf, study_start, study_end):
    """
    Derive epochs from the unique 'opened' dates in the infrastructure.
    Each epoch spans from one opening date to the next.
    Only dates within [study_start, study_end] generate epoch boundaries.
    Infrastructure opened before study_start is always-on from epoch 0.
    """
    # All unique dates that fall within the study window define new boundaries
    unique_dates = sorted(infra_gdf["opened"].dropna().unique())

    # Build boundary list: study_start + any mid-window openings
    boundaries = [study_start]
    for d in unique_dates:
        d = pd.Timestamp(d)
        if study_start < d < study_end and d not in boundaries:
            boundaries.append(d)
    boundaries = sorted(boundaries)

    # Each epoch runs from boundaries[i] to boundaries[i+1] (or study_end)
    epochs = []
    for i, start in enumerate(boundaries):
        end = boundaries[i + 1] if i + 1 < len(boundaries) else study_end
        epochs.append({"epoch": i, "start": start, "end": end})

    return pd.DataFrame(epochs)


EPOCHS = build_epochs(infra_gcc, STUDY_START, STUDY_END)
EPOCHS.to_csv('glasgow_epochs.csv', index=False)
print(f"{len(EPOCHS)} epochs from {infra_gcc['opened'].nunique()} unique opening dates")
print(EPOCHS.to_string())


def infra_at_epoch(gdf, epoch_start):
    """Return only segments that were open at or before epoch_start."""
    return gdf[gdf["opened"] <= epoch_start]


# ── Compute exposure for each epoch ──
cov_parts = []
for _, ep in EPOCHS.iterrows():
    infra_ep = infra_at_epoch(infra_gcc, ep["start"])
    buffers_ep = build_infra_buffers(infra_ep, INFRA_TYPES, ANY_TYPES)
    e = exposure_for_routes(routes, buffers_ep, INFRA_TYPES)
    e["epoch"] = int(ep["epoch"])
    cov_parts.append(e)
    print(f"epoch {ep['epoch']:2d}  starts {ep['start'].date()}  "
          f"segments={len(infra_ep):4d}  pairs={len(e)}")

cov = pd.concat(cov_parts, ignore_index=True)
cov.to_csv("glasgow_exposure_by_epoch.csv", index=False)
print(f"\nSaved glasgow_exposure_by_epoch.csv  ({len(cov):,} rows)")


22 epochs from 26 unique opening dates
    epoch      start        end
0       0 2017-09-15 2018-08-01
1       1 2018-08-01 2018-11-14
2       2 2018-11-14 2019-03-29
3       3 2019-03-29 2020-05-15
4       4 2020-05-15 2020-07-17
5       5 2020-07-17 2020-07-21
6       6 2020-07-21 2020-08-20
7       7 2020-08-20 2020-09-01
8       8 2020-09-01 2020-09-13
9       9 2020-09-13 2020-09-29
10     10 2020-09-29 2020-10-02
11     11 2020-10-02 2020-12-04
12     12 2020-12-04 2021-02-22
13     13 2021-02-22 2021-03-21
14     14 2021-03-21 2021-04-10
15     15 2021-04-10 2021-04-27
16     16 2021-04-27 2021-05-01
17     17 2021-05-01 2021-06-06
18     18 2021-06-06 2023-02-01
19     19 2023-02-01 2023-04-03
20     20 2023-04-03 2023-07-06
21     21 2023-07-06 2024-04-01
epoch  0  starts 2017-09-15  segments= 817  pairs=5904
epoch  1  starts 2018-08-01  segments= 821  pairs=5904
epoch  2  starts 2018-11-14  segments= 829  pairs=5904
epoch  3  starts 2019-03-29  segments= 834  pairs=5904
epoch

## 9. Assign trips to epochs → panel + trip-weighted mean


In [11]:
def assign_epoch(ts, epochs=EPOCHS):
    """Vectorised: map each trip timestamp to its epoch index."""
    hit = ((ts.values[:, None] >= epochs["start"].values[None, :]) &
           (ts.values[:, None] <  epochs["end"].values[None, :]))
    idx = hit.argmax(axis=1)
    out = epochs["epoch"].values[idx].astype(float)
    out[~hit.any(axis=1)] = np.nan
    return out


t = trips.copy()
t["epoch"] = assign_epoch(t["started_at"])
t = t.dropna(subset=["epoch"]); t["epoch"] = t["epoch"].astype(int)
t["origin_c"], t["dest_c"] = zip(*[canon(o, d) for o, d in zip(t["origin"], t["destination"])])
exp_cols = [c for c in cov.columns if c.startswith("exp_") or c.startswith("len_")]

# ── Option A: PANEL (od × epoch) ──
od_panel = (t.groupby(["origin", "destination", "origin_c", "dest_c", "epoch"])
              .size().reset_index(name="trips")
              .merge(cov, on=["origin_c", "dest_c", "epoch"], how="left"))
od_panel.to_csv("glasgow_exposure_panel.csv", index=False)

# ── Option B: TRIP-WEIGHTED MEAN (one row per OD pair) ──
tw = t.merge(cov[["origin_c", "dest_c", "epoch"] + exp_cols],
             on=["origin_c", "dest_c", "epoch"], how="left")
od_twm = (tw.groupby(["origin", "destination"])
            .agg(trips=("epoch", "size"), **{c: (c, "mean") for c in exp_cols})
            .reset_index())
od_twm.to_csv("glasgow_exposure_twm.csv", index=False)

# ── Clip exposure proportions above 1.0 (snapping artefact) ──
for c in [col for col in exp_cols if col.startswith("exp_")]:
    od_twm[c] = od_twm[c].clip(upper=1.0)
    od_panel[c] = od_panel[c].clip(upper=1.0)

print(f"PANEL rows: {len(od_panel):,}  |  TWM rows: {len(od_twm):,}")
print(f"Trips in panel: {od_panel['trips'].sum():,}")
print(od_twm[["origin", "destination", "trips", "exp_any", "exp_segregated"]].head())


PANEL rows: 68,901  |  TWM rows: 11,062
Trips in panel: 1,201,991
                               origin                                        destination  trips   exp_any  exp_segregated
0  Alexandra Parade (West) - ELECTRIC  Alexandra Park (south entrance) Alexandra Para...    388  0.000000        0.000000
1  Alexandra Parade (West) - ELECTRIC                          Anderston Railway Station      2  0.113684        0.043786
2  Alexandra Parade (West) - ELECTRIC                                 Anniesland Library      2  0.026504        0.009067
3  Alexandra Parade (West) - ELECTRIC           Argyle Street Railway Station - ELECTRIC    263  0.030269        0.000000
4  Alexandra Parade (West) - ELECTRIC                             Barrowlands - ELECTRIC    316  0.000000        0.000000


In [12]:
import pandas as pd

cov_blocked = pd.read_csv('glasgow_exposure_by_epoch.csv')
cov_blocked = cov_blocked.sort_values(['origin_c', 'dest_c', 'epoch']).reset_index(drop=True)

# columns that define a distinct infrastructure "state" for a route
state_cols = ['len_segregated_m', 'len_lane_m', 'len_shared_m', 'len_any_m']

r = cov_blocked[state_cols].round(3)                       # mm precision — kills float noise
prev = cov_blocked.groupby(['origin_c', 'dest_c'])[state_cols].shift().round(3)
changed = (r != prev).any(axis=1) | prev.isna().all(axis=1)   # first row = always a new block

cov_blocked['block'] = changed.groupby([cov_blocked['origin_c'], cov_blocked['dest_c']]).cumsum()

In [13]:
epoch_dates = pd.read_csv('glasgow_epochs.csv', parse_dates=['start', 'end'])

block_span = (cov_blocked.groupby(['origin_c', 'dest_c', 'block'])['epoch']
                 .agg(epoch_start='min', epoch_end='max')
                 .reset_index())

block_span = block_span.merge(
    epoch_dates.rename(columns={'epoch': 'epoch_start', 'start': 'block_start'})[['epoch_start', 'block_start']],
    on='epoch_start'
).merge(
    epoch_dates.rename(columns={'epoch': 'epoch_end', 'end': 'block_end'})[['epoch_end', 'block_end']],
    on='epoch_end'
)

In [15]:
epoch_to_block = cov_blocked[['origin_c', 'dest_c', 'epoch', 'block']]
exp_cols = [c for c in cov_blocked.columns if c.startswith('exp_') or c.startswith('len_')]

od_panel_collapsed = (
    od_panel
    .merge(epoch_to_block, on=['origin_c', 'dest_c', 'epoch'], how='left')
    .groupby(['origin', 'destination', 'origin_c', 'dest_c', 'block'])
    .agg(trips=('trips', 'sum'), **{c: (c, 'first') for c in exp_cols})  # exposure identical within a block by construction
    .reset_index()
)

In [16]:
block_span.to_csv('glasgow_block_span.csv', index=False)
od_panel_collapsed.to_csv('glasgow_od_panel_collapsed.csv', index=False)

## Outputs
The Trip Weighted Mean file is the regression-ready exposure variable.
